# 00 · Setup and smoke test

Checks that the whole environment works end to end: Drive, repo, dependencies, dataset, GPU and W&B.

**Phase 0 is done when anyone on the team runs this notebook top to bottom without errors.**

Before the first run, follow the one-time setup in the README (Drive shortcut + Colab secrets `GH_TOKEN` and `WANDB_API_KEY`). In Colab: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# --- Bootstrap: same cell at the top of every notebook ---
import os
import subprocess
import sys
from pathlib import Path

REPO = "matifleman/inspectia"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/InspectIA")
    REPO_DIR = Path("/content/inspectia")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        url = f"https://{userdata.get('GH_TOKEN')}@github.com/{REPO}.git"
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE_ROOT = REPO_DIR
    if "WANDB_API_KEY" not in os.environ:
        os.environ["WANDB_MODE"] = "offline"  # local runs without a key log offline

os.chdir(REPO_DIR)
if IN_COLAB:  # locally, install once in your venv (see README)
    %pip install -q -r requirements.txt -e .
# Editable installs are only picked up after a kernel restart; this makes the
# package importable in the current session too.
sys.path.insert(0, str(REPO_DIR / "src"))

DATA_DIR = DRIVE_ROOT / "data"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
SPLITS_DIR = DRIVE_ROOT / "splits"

from inspectia.utils import get_device, set_seed

SEED = 42
set_seed(SEED)
device = get_device()
print(f"Repo: {REPO_DIR} | Data: {DATA_DIR} | Device: {device}")


## 1. Dataset

Downloads KolektorSDD2 to Drive the first time (~850 MB, a few minutes); later runs only verify the counts.

In [ ]:
!{sys.executable} scripts/download_data.py --dest "{DATA_DIR}"
DATASET_DIR = DATA_DIR / "KolektorSDD2"

## 2. Load a small batch

4 good and 4 defective parts from train. A part is defective when its `_GT` mask is not empty. The transform here is only for the smoke test; the real preprocessing is defined in Phase 1.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torchvision.transforms import v2

def is_defective(image_path: Path) -> bool:
    mask = np.array(Image.open(image_path.with_name(f"{image_path.stem}_GT.png")))
    return bool(mask.any())

images = sorted(p for p in (DATASET_DIR / "train").glob("*.png") if not p.stem.endswith("_GT"))
good, defective = [], []
for p in images:
    (defective if is_defective(p) else good).append(p)
    if len(good) >= 4 and len(defective) >= 4:
        break
sample = good[:4] + defective[:4]
labels = torch.tensor([0] * 4 + [1] * 4)

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize((640, 224)),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
batch = torch.stack([transform(Image.open(p).convert("RGB")) for p in sample])
print("Batch:", tuple(batch.shape), batch.dtype, "| labels:", labels.tolist())

fig, axes = plt.subplots(1, 8, figsize=(16, 6))
for ax, p, y in zip(axes, sample, labels):
    ax.imshow(Image.open(p))
    ax.set_title(f"{p.stem}\n{'DEFECT' if y else 'OK'}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. ResNet18 on the GPU

In [ ]:
from torchvision.models import ResNet18_Weights, resnet18

model = resnet18(weights=ResNet18_Weights.DEFAULT).to(device).eval()
with torch.no_grad():
    logits = model(batch.to(device))
print("Output:", tuple(logits.shape), "on", logits.device)
assert logits.shape == (8, 1000)

## 4. W&B

Logs a test run to the team project. Check that it shows up at wandb.ai.

In [ ]:
import wandb

run = wandb.init(project="inspectia", job_type="smoke-test", config={"seed": SEED, "device": str(device)})
run.log({"smoke/logit_mean": logits.mean().item(), "smoke/batch": wandb.Image(fig)})
run.finish()

## 5. Versions

Copy this output into the README table and `requirements.txt` pins (Phase 0, step 5).

In [ ]:
import importlib.metadata as md
import platform

print("python", platform.python_version())
for pkg in ["torch", "torchvision", "numpy", "pandas", "scikit-learn", "matplotlib",
            "seaborn", "pyyaml", "pillow", "tqdm", "wandb"]:
    print(pkg, md.version(pkg))
print("cuda", torch.version.cuda, "| gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)